# 🔗 Sparse Feature Circuits (Marks et al. 2024)

Replication of **"Sparse Feature Circuits: Discovering and Editing Interpretable Causal Graphs in Language Models"** — Marks, Rager, Michaud, Belinkov, Bau, Mueller (2024).

- Paper: [arXiv:2403.19647](https://arxiv.org/abs/2403.19647)
- HTML v3 (with Appendix A.1 edge formula): https://arxiv.org/html/2403.19647v3
- Reference code: https://github.com/saprmarks/feature-circuits (MIT)

## What this notebook ships

1. **Node attribution** via **AtP** (Attribution Patching) with mean ablation:
   $$\text{IE}_\text{node}^{\text{atp}}(f_i) = \nabla_{z_i} m \cdot (a^\text{patch}_i - a^\text{clean}_i)$$
2. **Integrated Gradients** fallback for early layers (0, 1 MLP) with N=10 steps:
   $$\text{IE}_\text{ig}(f_i) = \frac{1}{N}\sum_{\alpha \in [0,1]} \nabla_{z_i} m \big|_\alpha \cdot (a^\text{patch}_i - a^\text{clean}_i)$$
3. **Edge attribution** (Appendix A.1) between adjacent SAE layers $\ell$ and $\ell+1$:
   $$\text{IE}_\text{edge}(u\!\to\!d) = \nabla_{z_d} m \cdot W^\text{enc}_d \cdot W^\text{dec}_u[:, i_u] \cdot (a^\text{patch}_u - a^\text{clean}_u)$$
4. **SAE error terms** $\varepsilon = x - \text{SAE}(x)$ are triangle nodes in the DAG and are also scored.
5. Thresholding: $|\text{IE}| > \tau_\text{node}$ (default 0.1) for nodes, $|\text{IE}| > \tau_\text{edge}$ (default 0.01) for edges.

Output: a `circuit.json` DAG consumable by the Circuit Canvas React viewer at `/observatory/circuits`.

In [ ]:
# Install dependencies
!pip -q install --upgrade transformers accelerate safetensors huggingface_hub
!pip -q install networkx matplotlib tqdm

## Config

Two SAEs on adjacent (or pair of) layers of the same base model. The notebook builds a 2-layer feature circuit between them. For a 3-layer circuit, stack this notebook twice.

In [ ]:
# === Config ===
HF_BASE_MODEL = "Qwen/Qwen3-27B"   # base LM for both SAEs

# Upstream SAE (earlier layer, e.g. L11)
SAE_UPSTREAM_REPO   = "caiovicentino1/qwen36-27b-sae-multilayer"
SAE_UPSTREAM_SUBDIR = "L11"          # HF subdir holding the L11 SAE
UPSTREAM_LAYER      = 11

# Downstream SAE (later layer, e.g. L31)
SAE_DOWNSTREAM_REPO   = "caiovicentino1/qwen36-27b-sae-multilayer"
SAE_DOWNSTREAM_SUBDIR = "L31"
DOWNSTREAM_LAYER      = 31

# SAE architecture (TopK)
D_MODEL = 3584    # residual-stream width of the base model (adjust)
D_SAE   = 65536   # SAE latent width (adjust to whatever was trained)
K       = 64      # TopK

# Prompts (paired clean / patch pairs used for AtP-style mean-ablation):
# If you only have a flat prompt list, the patch activation becomes the
# batch-wise mean over the prompt set (classic "mean ablation").
PROMPTS = [
    "The patient presented with severe chest pain and shortness of breath.",
    "The algorithm runs in O(n log n) time using a balanced binary search tree.",
    "The treaty was signed in 1648 and marked the end of the Thirty Years' War.",
    "She multiplied 17 by 23 in her head and got 391.",
    "The doctor prescribed 500 mg of amoxicillin twice daily for ten days.",
    "To compile the kernel, run `make -j$(nproc)` in the source directory.",
    "The Fourier transform of a Gaussian is another Gaussian.",
    "In Python, a dict is an ordered hash map as of 3.7.",
    "The French Revolution began in 1789 with the storming of the Bastille.",
    "A p-value of 0.04 is conventionally considered statistically significant.",
    "The mitochondrion is the powerhouse of the cell.",
    "The derivative of sin(x) with respect to x is cos(x).",
    "git rebase -i HEAD~3 lets you squash or reorder the last three commits.",
    "The Pythagorean theorem relates the three sides of a right triangle.",
    "Hemoglobin carries oxygen from the lungs to peripheral tissues.",
    "The capital of Australia is Canberra, not Sydney.",
    "Entropy in information theory measures the average surprise of a distribution.",
    "A transformer block contains self-attention followed by a feed-forward MLP.",
    "Shakespeare wrote Hamlet around the year 1600.",
    "The speed of light in vacuum is approximately 2.998 x 10^8 m/s.",
]

# Thresholds (Marks et al. sweep these; defaults are conservative)
TAU_NODE = 0.1
TAU_EDGE = 0.01

# Integrated Gradients — use for early layers only (layers 0, 1 MLP)
USE_IG_EARLY = True
IG_STEPS     = 10   # set to 1 on T4 / low-VRAM
EARLY_LAYERS_FOR_IG = {0, 1}

# Target metric for backward pass
TARGET_METRIC     = "logit"   # "logit" or "logprob"
TARGET_TOKEN_IDX  = -1         # which position (default = last) drives m

# Runtime / storage
HF_PUSH_REPO = "caiovicentino1/qwen36-27b-feature-circuit-L11-L31"  # set to None to skip push
OUT_PATH     = "circuit.json"

import torch
DTYPE = torch.bfloat16
print(f"Config loaded: {UPSTREAM_LAYER} -> {DOWNSTREAM_LAYER}, d_model={D_MODEL}, d_sae={D_SAE}, K={K}, N_prompts={len(PROMPTS)}")

## HuggingFace auth

In [ ]:
import os
try:
    from google.colab import userdata  # Colab
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # assume env var already set

from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"), add_to_git_credential=False)
print("HF login OK")

## Load base model + both SAEs

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

device = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    dtype=DTYPE,
    attn_implementation="sdpa",
    device_map="auto",
    trust_remote_code=True,
).eval()

# Resolve the submodule list (.layers path differs per arch)
def _layers_module(m):
    if hasattr(m, "model") and hasattr(m.model, "layers"):
        return m.model.layers
    if hasattr(m, "model") and hasattr(m.model, "language_model") and hasattr(m.model.language_model, "layers"):
        return m.model.language_model.layers
    if hasattr(m, "language_model") and hasattr(m.language_model, "layers"):
        return m.language_model.layers
    raise RuntimeError("Could not locate decoder .layers for this arch")

LAYERS = _layers_module(model)
print(f"Base model loaded; found {len(LAYERS)} decoder layers")


# ---- TopK SAE ----
class TopKSAE(nn.Module):
    def __init__(self, d_model, d_sae, k):
        super().__init__()
        self.d_model = d_model
        self.d_sae   = d_sae
        self.k       = k
        self.W_enc = nn.Parameter(torch.zeros(d_model, d_sae))     # x @ W_enc -> pre-acts
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.zeros(d_sae, d_model))     # z @ W_dec -> reconstruction
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        topv, topi = torch.topk(pre, self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, topi, torch.relu(topv))
        return z

    def decode(self, z):
        return z @ self.W_dec + self.b_dec

    def forward(self, x):
        z       = self.encode(x)
        x_hat   = self.decode(z)
        err     = x - x_hat
        return z, x_hat, err


def _load_sae(repo, subdir, d_model, d_sae, k):
    # Try canonical filenames
    candidates = [f"{subdir}/sae.safetensors", f"{subdir}/model.safetensors",
                  f"{subdir}/weights.safetensors", f"{subdir}/pytorch_model.bin"]
    state = None
    for fn in candidates:
        try:
            p = hf_hub_download(repo_id=repo, filename=fn)
            state = load_file(p) if p.endswith(".safetensors") else torch.load(p, map_location="cpu")
            print(f"Loaded SAE weights: {repo}/{fn}")
            break
        except Exception as e:
            continue
    if state is None:
        raise RuntimeError(f"Could not download SAE weights from {repo}/{subdir}")
    sae = TopKSAE(d_model, d_sae, k)
    missing, unexpected = sae.load_state_dict(state, strict=False)
    if missing:    print("  missing:",    missing[:4], "..." if len(missing)>4 else "")
    if unexpected: print("  unexpected:", unexpected[:4], "..." if len(unexpected)>4 else "")
    return sae.to(device=device, dtype=DTYPE).eval()


sae_up   = _load_sae(SAE_UPSTREAM_REPO,   SAE_UPSTREAM_SUBDIR,   D_MODEL, D_SAE, K)
sae_down = _load_sae(SAE_DOWNSTREAM_REPO, SAE_DOWNSTREAM_SUBDIR, D_MODEL, D_SAE, K)
print("Both SAEs on device, dtype =", DTYPE)

## Hooks for dual-layer capture

We hook the residual stream at the output of `layers[UPSTREAM_LAYER]` and `layers[DOWNSTREAM_LAYER]`, run each through its SAE to obtain `z`, `x_hat`, `err`, and splice `x_hat + err` back into the residual stream so the forward pass is unchanged but `z` and `err` carry gradients.

In [ ]:
class SAECapture:
    """Forward hook that: (a) runs SAE on residual, (b) replaces residual with x_hat+err
    so that z and err are part of the differentiable graph reaching the logits."""

    def __init__(self, sae: TopKSAE, scale_z: torch.Tensor | None = None):
        self.sae = sae
        # scale_z: optional per-feature multiplier in (0,1], used for IG alpha interpolation
        self.scale_z = scale_z
        self.z   = None  # (B, T, d_sae)  -- retain_grad
        self.err = None  # (B, T, d_model) -- retain_grad

    def hook(self, module, inputs, output):
        # HF decoder layers return either a tensor or a tuple (hidden, ...)
        if isinstance(output, tuple):
            hidden = output[0]
            rest   = output[1:]
        else:
            hidden = output
            rest   = None
        x = hidden
        z = self.sae.encode(x)
        if self.scale_z is not None:
            z = z * self.scale_z  # IG alpha-scaled reconstruction
        x_hat = self.sae.decode(z)
        err   = x - self.sae.decode(self.sae.encode(x))  # clean error (detach-safe below)
        # Make z and err leafy-ish for .grad
        z   = z.detach().clone().requires_grad_(True)       if False else z
        err = err.detach().clone().requires_grad_(True)     if False else err
        z.retain_grad()
        err.retain_grad()
        self.z, self.err = z, err
        # Splice back: reconstruction through SAE + error -> identity on forward, but
        # gradients flow through z and err independently.
        new_hidden = self.sae.decode(z) + err
        if rest is None:
            return new_hidden
        return (new_hidden,) + rest

cap_up   = SAECapture(sae_up)
cap_down = SAECapture(sae_down)

def install_hooks():
    h1 = LAYERS[UPSTREAM_LAYER  ].register_forward_hook(cap_up.hook)
    h2 = LAYERS[DOWNSTREAM_LAYER].register_forward_hook(cap_down.hook)
    return [h1, h2]

print("Hooks defined for layers", UPSTREAM_LAYER, DOWNSTREAM_LAYER)

## Node attribution (AtP + IG fallback)

For each prompt we:
1. Forward pass, capture `z_up`, `err_up`, `z_down`, `err_down`.
2. Pick metric `m = logit[TARGET_TOKEN_IDX, argmax_clean]` (or logprob).
3. Backward pass `m.backward()` → grads on `z_*` and `err_*`.
4. `IE_node(f_i) = grad_z[i] * (a_patch_i - a_clean_i)`, with `a_patch = mean_over_prompts(z)`.
5. For each layer in `EARLY_LAYERS_FOR_IG`, replace AtP with Integrated Gradients (IG_STEPS interpolations).

Activations are aggregated to scalar per feature by summing over sequence positions (Marks et al. §3.1).

In [ ]:
from tqdm.auto import tqdm

def _metric(logits, target_idx):
    # logits: (B, T, V).  Take position TARGET_TOKEN_IDX, reduce to scalar = argmax logit.
    l = logits[:, TARGET_TOKEN_IDX, :]
    if TARGET_METRIC == "logprob":
        l = torch.log_softmax(l.float(), dim=-1)
    # scalar = logit of argmax token, summed across batch
    top = l.argmax(dim=-1)
    m = l.gather(-1, top.unsqueeze(-1)).squeeze(-1).sum()
    return m

@torch.no_grad()
def _collect_clean_activations(prompt_ids):
    """One clean forward pass, store z and err on CPU to save VRAM."""
    handles = install_hooks()
    try:
        _ = model(prompt_ids)
        z_up   = cap_up.z.detach().to("cpu")
        err_up = cap_up.err.detach().to("cpu")
        z_dn   = cap_down.z.detach().to("cpu")
        err_dn = cap_down.err.detach().to("cpu")
    finally:
        for h in handles: h.remove()
    return z_up, err_up, z_dn, err_dn


def _atp_grads(prompt_ids):
    """Forward with grad on, backward the metric, return grads on z_up/err_up/z_dn/err_dn."""
    handles = install_hooks()
    try:
        for p in model.parameters():
            p.requires_grad_(False)
        out = model(prompt_ids)
        logits = out.logits if hasattr(out, "logits") else out[0]
        m = _metric(logits, TARGET_TOKEN_IDX)
        model.zero_grad(set_to_none=True)
        m.backward()
        g_zu  = cap_up.z.grad.detach()   if cap_up.z.grad   is not None else torch.zeros_like(cap_up.z)
        g_eu  = cap_up.err.grad.detach() if cap_up.err.grad is not None else torch.zeros_like(cap_up.err)
        g_zd  = cap_down.z.grad.detach()   if cap_down.z.grad   is not None else torch.zeros_like(cap_down.z)
        g_ed  = cap_down.err.grad.detach() if cap_down.err.grad is not None else torch.zeros_like(cap_down.err)
        return (cap_up.z.detach(), cap_up.err.detach(), cap_down.z.detach(), cap_down.err.detach(),
                g_zu, g_eu, g_zd, g_ed, m.detach())
    finally:
        for h in handles: h.remove()


def _ig_grads(prompt_ids, which: str):
    """Integrated Gradients for early layer. `which` in {'up','down'}."""
    cap = cap_up if which == "up" else cap_down
    alphas = torch.linspace(1.0/IG_STEPS, 1.0, IG_STEPS, device=device, dtype=DTYPE)
    grads_sum = None
    z_clean_ref = None
    err_clean_ref = None
    for a in alphas:
        cap.scale_z = a  # broadcast scalar
        handles = install_hooks()
        try:
            out = model(prompt_ids)
            logits = out.logits if hasattr(out, "logits") else out[0]
            m = _metric(logits, TARGET_TOKEN_IDX)
            model.zero_grad(set_to_none=True)
            m.backward()
            g = cap.z.grad.detach()
            if grads_sum is None:
                grads_sum     = g.clone()
                z_clean_ref   = cap.z.detach().clone() / a  # undo scale to recover base z
                err_clean_ref = cap.err.detach().clone()
            else:
                grads_sum += g
        finally:
            for h in handles: h.remove()
    cap.scale_z = None
    return z_clean_ref, err_clean_ref, grads_sum / IG_STEPS


def compute_node_scores():
    # ---- Pass 1: mean ("patch") activation over prompts ----
    # Store per-prompt CPU-side, then average.
    sum_zu = torch.zeros(D_SAE,   dtype=torch.float32)
    sum_eu = torch.zeros(D_MODEL, dtype=torch.float32)
    sum_zd = torch.zeros(D_SAE,   dtype=torch.float32)
    sum_ed = torch.zeros(D_MODEL, dtype=torch.float32)
    n_tok = 0

    tokenized = [tok(p, return_tensors="pt").input_ids.to(device) for p in PROMPTS]

    for ids in tqdm(tokenized, desc="clean pass (patch=mean)"):
        z_up, err_up, z_dn, err_dn = _collect_clean_activations(ids)
        # sum over (B, T) -> per-feature scalar
        sum_zu += z_up.float().sum(dim=(0,1)).cpu()
        sum_eu += err_up.float().sum(dim=(0,1)).cpu()
        sum_zd += z_dn.float().sum(dim=(0,1)).cpu()
        sum_ed += err_dn.float().sum(dim=(0,1)).cpu()
        n_tok  += z_up.shape[0] * z_up.shape[1]

    a_patch_zu = (sum_zu / max(n_tok,1)).to(device=device, dtype=DTYPE)  # (d_sae,)
    a_patch_eu = (sum_eu / max(n_tok,1)).to(device=device, dtype=DTYPE)
    a_patch_zd = (sum_zd / max(n_tok,1)).to(device=device, dtype=DTYPE)
    a_patch_ed = (sum_ed / max(n_tok,1)).to(device=device, dtype=DTYPE)

    # ---- Pass 2: AtP (or IG) node scores ----
    ie_zu = torch.zeros(D_SAE,   dtype=torch.float32)
    ie_eu = torch.zeros(D_MODEL, dtype=torch.float32)
    ie_zd = torch.zeros(D_SAE,   dtype=torch.float32)
    ie_ed = torch.zeros(D_MODEL, dtype=torch.float32)

    # Cache upstream state for reuse in edge attribution
    cache = {
        "z_up_per_prompt":   [],
        "g_zd_per_prompt":   [],
        "a_patch_zu":        a_patch_zu.detach().cpu(),
    }

    use_ig_up   = USE_IG_EARLY and (UPSTREAM_LAYER   in EARLY_LAYERS_FOR_IG)
    use_ig_down = USE_IG_EARLY and (DOWNSTREAM_LAYER in EARLY_LAYERS_FOR_IG)

    for ids in tqdm(tokenized, desc="backward pass (AtP/IG)"):
        if use_ig_up or use_ig_down:
            # IG on the early layer, AtP on the other via one clean bwd pass.
            zu, eu, zd, ed, g_zu_atp, g_eu, g_zd_atp, g_ed, _m = _atp_grads(ids)
            g_zu = g_zu_atp
            g_zd = g_zd_atp
            if use_ig_up:
                zu_ig, eu_ig, g_zu_ig = _ig_grads(ids, which="up")
                g_zu = g_zu_ig
            if use_ig_down:
                zd_ig, ed_ig, g_zd_ig = _ig_grads(ids, which="down")
                g_zd = g_zd_ig
        else:
            zu, eu, zd, ed, g_zu, g_eu, g_zd, g_ed, _m = _atp_grads(ids)

        # IE_i = grad_i * (a_patch_i - a_clean_i)  — sum over seq positions
        delta_zu = (a_patch_zu - zu).float()
        delta_eu = (a_patch_eu - eu).float()
        delta_zd = (a_patch_zd - zd).float()
        delta_ed = (a_patch_ed - ed).float()

        ie_zu += (g_zu.float() * delta_zu).sum(dim=(0,1)).cpu()
        ie_eu += (g_eu.float() * delta_eu).sum(dim=(0,1)).cpu()
        ie_zd += (g_zd.float() * delta_zd).sum(dim=(0,1)).cpu()
        ie_ed += (g_ed.float() * delta_ed).sum(dim=(0,1)).cpu()

        # Store for edge attribution (CPU to save VRAM)
        cache["z_up_per_prompt"].append(zu.detach().cpu())
        cache["g_zd_per_prompt"].append(g_zd.detach().cpu())

        # Explicit cleanup
        del zu, eu, zd, ed, g_zu, g_eu, g_zd, g_ed
        torch.cuda.empty_cache()

    # Normalize by prompt count
    N = len(PROMPTS)
    ie_zu /= N; ie_eu /= N; ie_zd /= N; ie_ed /= N

    return {
        "ie_zu": ie_zu, "ie_eu": ie_eu,
        "ie_zd": ie_zd, "ie_ed": ie_ed,
        "cache": cache,
    }


node_scores = compute_node_scores()
print("Node attribution done.")
print(f"  upstream  features |IE|>tau: {(node_scores['ie_zu'].abs()>TAU_NODE).sum().item()}")
print(f"  upstream  err      |IE|>tau: {(node_scores['ie_eu'].abs()>TAU_NODE).sum().item()} dims")
print(f"  downstream features |IE|>tau: {(node_scores['ie_zd'].abs()>TAU_NODE).sum().item()}")
print(f"  downstream err      |IE|>tau: {(node_scores['ie_ed'].abs()>TAU_NODE).sum().item()} dims")

## Edge attribution (Appendix A.1)

For an upstream feature $u$ at layer $\ell$ with decoder row $W^\text{dec}_u[i_u, :]$ (shape $d_\text{model}$) and a downstream feature $d$ at layer $\ell+1$ with encoder column $W^\text{enc}_d[:, d]$ (shape $d_\text{model}$), the edge indirect effect is

$$\text{IE}_\text{edge}(u\!\to\!d) = \underbrace{\nabla_{z_d} m}_{\text{grad at downstream feature }d} \;\cdot\; \underbrace{W^\text{enc}_{d}[:, d]^\top \cdot W^\text{dec}_{u}[i_u, :]^\top}_{\text{linear chain }\ell\!\to\!\ell\!+\!1} \;\cdot\; \underbrace{(a^\text{patch}_u - a^\text{clean}_u)}_{\text{upstream delta}}$$

We restrict to upstream/downstream features that survived the node threshold, which keeps the per-pair cost under control.

In [ ]:
def compute_edge_scores(node_scores, tau_node=TAU_NODE):
    ie_zu = node_scores["ie_zu"]
    ie_zd = node_scores["ie_zd"]
    cache = node_scores["cache"]

    up_keep   = (ie_zu.abs() > tau_node).nonzero(as_tuple=False).squeeze(-1).tolist()
    down_keep = (ie_zd.abs() > tau_node).nonzero(as_tuple=False).squeeze(-1).tolist()
    print(f"Edge pass: |U|={len(up_keep)} upstream x |D|={len(down_keep)} downstream = {len(up_keep)*len(down_keep)} candidate edges")

    if not up_keep or not down_keep:
        return {"edges": [], "up_keep": up_keep, "down_keep": down_keep}

    # W_dec_up: (d_sae, d_model) — rows are feature decoder directions
    # W_enc_dn: (d_model, d_sae) — columns are feature encoder directions
    W_dec_u = sae_up.W_dec.detach()           # (d_sae, d_model)
    W_enc_d = sae_down.W_enc.detach()         # (d_model, d_sae)

    # Slice to surviving subset
    up_idx   = torch.tensor(up_keep,   device=device, dtype=torch.long)
    down_idx = torch.tensor(down_keep, device=device, dtype=torch.long)
    Wd_u_sub = W_dec_u.index_select(0, up_idx)       # (|U|, d_model)
    We_d_sub = W_enc_d.index_select(1, down_idx)     # (d_model, |D|)

    # Linear-chain matrix L = Wd_u_sub @ We_d_sub   -> (|U|, |D|)
    L = (Wd_u_sub.float() @ We_d_sub.float())

    # Accumulate per-prompt contribution
    #   IE_edge[u,d] = mean_prompt [ g_zd[:, :, d].sum(seq) * L[u,d] * (a_patch_u - z_up[:, :, u].sum(seq)) ]
    # Note: aggregate over sequence positions as in Marks et al. (§3.1 + A.1).
    ie_edge = torch.zeros(len(up_keep), len(down_keep), dtype=torch.float32)
    a_patch_u_sub = node_scores["cache"]["a_patch_zu"].index_select(0, torch.tensor(up_keep)).float()  # (|U|,)

    for zu_cpu, gzd_cpu in zip(cache["z_up_per_prompt"], cache["g_zd_per_prompt"]):
        # zu_cpu  : (B, T, d_sae)   — clean upstream activations
        # gzd_cpu : (B, T, d_sae)   — grad at downstream features
        zu_sub  = zu_cpu.float().index_select(-1, torch.tensor(up_keep)).sum(dim=(0,1))       # (|U|,)
        gzd_sub = gzd_cpu.float().index_select(-1, torch.tensor(down_keep)).sum(dim=(0,1))   # (|D|,)
        delta_u = (a_patch_u_sub - zu_sub)                                                   # (|U|,)
        # outer product: (|U|, |D|)
        contrib = (delta_u.unsqueeze(1) * gzd_sub.unsqueeze(0)) * L.cpu()
        ie_edge += contrib

    ie_edge /= max(len(cache["z_up_per_prompt"]), 1)

    # Threshold
    edges = []
    mask = ie_edge.abs() > TAU_EDGE
    rows, cols = mask.nonzero(as_tuple=True)
    for r, c in zip(rows.tolist(), cols.tolist()):
        edges.append({
            "u":     up_keep[r],
            "d":     down_keep[c],
            "score": float(ie_edge[r, c].item()),
        })
    edges.sort(key=lambda e: abs(e["score"]), reverse=True)
    print(f"Edges surviving |IE|>{TAU_EDGE}: {len(edges)}")
    return {"edges": edges, "up_keep": up_keep, "down_keep": down_keep, "ie_edge": ie_edge}


edge_scores = compute_edge_scores(node_scores, tau_node=TAU_NODE)

## Build DAG JSON and publish

Schema (contract with the Circuit Canvas React viewer at `/observatory/circuits`):

```json
{
  "nodes": [
    {"id": "up.f123", "layer": "L11", "score": 0.42, "kind": "feature", "name": "overconfidence_pattern"},
    {"id": "up.error", "layer": "L11", "score": 0.15, "kind": "error"},
    {"id": "down.f456", "layer": "L31", "score": 0.38, "kind": "feature", "name": "medical_terms"}
  ],
  "edges": [
    {"source": "up.f123", "target": "down.f456", "score": 0.08}
  ],
  "metric": "logit",
  "prompts_n": 20,
  "tau_node": 0.1,
  "tau_edge": 0.01
}
```

In [ ]:
import json
from huggingface_hub import HfApi, create_repo

def build_dag(node_scores, edge_scores, tau_node=TAU_NODE, tau_edge=TAU_EDGE):
    ie_zu, ie_eu = node_scores["ie_zu"], node_scores["ie_eu"]
    ie_zd, ie_ed = node_scores["ie_zd"], node_scores["ie_ed"]

    nodes = []

    # upstream features
    for i in (ie_zu.abs() > tau_node).nonzero(as_tuple=False).squeeze(-1).tolist():
        nodes.append({
            "id":    f"up.f{i}",
            "layer": f"L{UPSTREAM_LAYER}",
            "score": float(ie_zu[i].item()),
            "kind":  "feature",
            "name":  f"up_L{UPSTREAM_LAYER}_f{i}",
        })
    # upstream SAE error — aggregate the error-node score as L2 over its d_model dims
    eu_agg = float(ie_eu.abs().sum().item())
    if eu_agg > tau_node:
        nodes.append({
            "id":    "up.error",
            "layer": f"L{UPSTREAM_LAYER}",
            "score": eu_agg,
            "kind":  "error",
        })
    # downstream features
    for i in (ie_zd.abs() > tau_node).nonzero(as_tuple=False).squeeze(-1).tolist():
        nodes.append({
            "id":    f"down.f{i}",
            "layer": f"L{DOWNSTREAM_LAYER}",
            "score": float(ie_zd[i].item()),
            "kind":  "feature",
            "name":  f"down_L{DOWNSTREAM_LAYER}_f{i}",
        })
    # downstream SAE error
    ed_agg = float(ie_ed.abs().sum().item())
    if ed_agg > tau_node:
        nodes.append({
            "id":    "down.error",
            "layer": f"L{DOWNSTREAM_LAYER}",
            "score": ed_agg,
            "kind":  "error",
        })

    edges = [
        {
            "source": f"up.f{e['u']}",
            "target": f"down.f{e['d']}",
            "score":  e["score"],
        }
        for e in edge_scores.get("edges", [])
    ]

    return {
        "nodes":        nodes,
        "edges":        edges,
        "metric":       TARGET_METRIC,
        "prompts_n":    len(PROMPTS),
        "tau_node":     tau_node,
        "tau_edge":     tau_edge,
        "upstream_layer":   f"L{UPSTREAM_LAYER}",
        "downstream_layer": f"L{DOWNSTREAM_LAYER}",
        "base_model":       HF_BASE_MODEL,
        "sae_upstream":     f"{SAE_UPSTREAM_REPO}/{SAE_UPSTREAM_SUBDIR}",
        "sae_downstream":   f"{SAE_DOWNSTREAM_REPO}/{SAE_DOWNSTREAM_SUBDIR}",
        "ig_steps":         IG_STEPS if USE_IG_EARLY else 0,
    }


dag = build_dag(node_scores, edge_scores)

with open(OUT_PATH, "w") as f:
    json.dump(dag, f, indent=2)

# Summary
n_feat_nodes  = sum(1 for n in dag["nodes"] if n["kind"] == "feature")
n_error_nodes = sum(1 for n in dag["nodes"] if n["kind"] == "error")
mean_node_score = sum(abs(n["score"]) for n in dag["nodes"]) / max(len(dag["nodes"]), 1)
mean_edge_score = sum(abs(e["score"]) for e in dag["edges"]) / max(len(dag["edges"]), 1)

print(f"circuit.json written to {OUT_PATH}")
print(f"  nodes: {len(dag['nodes'])} ({n_feat_nodes} feature + {n_error_nodes} error)")
print(f"  edges: {len(dag['edges'])}")
print(f"  mean |IE_node| = {mean_node_score:.4f}")
print(f"  mean |IE_edge| = {mean_edge_score:.4f}")

# Push to HF
if HF_PUSH_REPO:
    api = HfApi()
    try:
        create_repo(HF_PUSH_REPO, repo_type="model", exist_ok=True, private=False)
    except Exception as e:
        print("create_repo:", e)
    api.upload_file(
        path_or_fileobj=OUT_PATH,
        path_in_repo="circuit.json",
        repo_id=HF_PUSH_REPO,
        repo_type="model",
    )
    print(f"Uploaded to https://huggingface.co/{HF_PUSH_REPO}/blob/main/circuit.json")

## Visualize preview

Local matplotlib preview of the DAG. The real interactive UI lives at `/observatory/circuits` in the web app — this is just for sanity-checking the JSON before upload.

- Node size ∝ $|\text{score}|$
- Edge thickness ∝ $|\text{score}|$
- Upstream features on the left, downstream on the right, error nodes drawn as triangles.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
for n in dag["nodes"]:
    G.add_node(n["id"], **n)
for e in dag["edges"]:
    G.add_edge(e["source"], e["target"], score=e["score"])

# Bipartite-style layout: upstream on x=0, downstream on x=1
pos = {}
up_nodes   = [n for n in G.nodes if n.startswith("up.")]
down_nodes = [n for n in G.nodes if n.startswith("down.")]
for i, n in enumerate(up_nodes):
    pos[n] = (0.0, -i * 1.0)
for i, n in enumerate(down_nodes):
    pos[n] = (1.5, -i * 1.0)

feat_nodes  = [n for n, d in G.nodes(data=True) if d.get("kind") == "feature"]
error_nodes = [n for n, d in G.nodes(data=True) if d.get("kind") == "error"]
feat_sizes  = [max(abs(G.nodes[n]["score"]) * 2000, 80) for n in feat_nodes]
error_sizes = [max(abs(G.nodes[n]["score"]) * 2000, 80) for n in error_nodes]

edge_widths = [max(abs(G.edges[u, v]["score"]) * 120, 0.3) for u, v in G.edges]
edge_colors = [("#d04040" if G.edges[u,v]["score"] < 0 else "#3070d0") for u, v in G.edges]

fig, ax = plt.subplots(figsize=(10, max(6, 0.4 * max(len(up_nodes), len(down_nodes)))))
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color=edge_colors, alpha=0.7, ax=ax, arrows=True)
nx.draw_networkx_nodes(G, pos, nodelist=feat_nodes,  node_size=feat_sizes,  node_color="#cfe3ff", edgecolors="#1f3a6a", node_shape="o", ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=error_nodes, node_size=error_sizes, node_color="#ffd9a8", edgecolors="#8a4b00", node_shape="^", ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
ax.set_title(f"Feature circuit  {dag['upstream_layer']} -> {dag['downstream_layer']}   (nodes={len(dag['nodes'])}, edges={len(dag['edges'])})")
ax.axis("off")
plt.tight_layout()
plt.savefig("circuit_preview.png", dpi=140, bbox_inches="tight")
plt.show()
print("Saved circuit_preview.png")